In [1]:
!pip install pandas matplotlib numpy openpyxl


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import os

In [3]:
df = pd.read_excel('../data/raw/data.xlsx', sheet_name='Sheet1')

In [ ]:
print("Nr of Observations before Cleaning: ", len(df))

df["timestamp"]     = pd.to_datetime(df["timestamp"])
df                  = df.sort_values("timestamp").reset_index(drop=True)
first_complete_idx  = df.notna().all(axis=1).idxmax()
df_cleaned          = df.iloc[first_complete_idx:].reset_index(drop=True)

print("Nr of Observations after trimming leading rows: ", len(df_cleaned))

# Invalidate the isolated FT-801 sentinel readings identified during EDA (-800.000000)
sentinel_mask = df_cleaned["FT-801"] == -800.0
print(f"Invalidating {sentinel_mask.sum()} FT-801 sentinel readings")
df_cleaned.loc[sentinel_mask, "FT-801"] = np.nan

# Flag the frozen PT-903 sensor episode (kept as-is, not removed or corrected),
# to serve as a known false-positive reference case for anomaly detection evaluation
frozen_mask = np.isclose(df_cleaned["PT-903"], -2.896296, atol=1e-4)
df_cleaned["pt903_frozen_flag"] = frozen_mask.astype(int)
print(f"Flagged {frozen_mask.sum()} observations as PT-903 frozen episode")

print("Nr of Observations after Cleaning: ", len(df_cleaned))

Nr of Observations before Cleaning:  40773
Nr of Observations after trimming leading rows:  39175
Invalidating 27 FT-801 sentinel readings
Flagged 294 observations as PT-903 frozen episode


In [5]:
df_cleaned["time_diff"] = df_cleaned["timestamp"].diff()
gaps = df_cleaned[df_cleaned["time_diff"] > pd.Timedelta(minutes=1)]

if not gaps.empty:
    total_time_lost = gaps["time_diff"].sum()
    print(f"Total time missing: {total_time_lost}")
    print(f"Nr of jumps found: {len(gaps)}")

Total time missing: 1 days 16:03:01.430000
Nr of jumps found: 77


In [ ]:
# Filter by the runmode
df_ops = df_cleaned[df_cleaned["Mode"].isin([8, 9])].copy()
df_ops = df_ops.sort_values("timestamp")

# Fill LS-901 globally (across gaps) before splitting into sessions,
# since oil level does not reset simply because the sensor did not report
df_ops["LS-901"] = df_ops["LS-901"].ffill().bfill()

# Identify the break points
is_big_gap = df_ops["timestamp"].diff() > pd.Timedelta(minutes=30)
df_ops["bucket_id"] = is_big_gap.cumsum()

all_buckets = []

for bid, group in df_ops.groupby("bucket_id"):

    group = group.set_index("timestamp").sort_index()

    # Interpolate the data for small gaps (including the invalidated FT-801 sentinel values)
    group_interp = group.resample('min').asfreq()
    group_interp = group_interp.interpolate(method='linear')

    group_interp["Mode"] = group_interp["Mode"].ffill()
    group_interp["LS-901"] = group_interp["LS-901"].ffill().astype(int)
    group_interp["pt903_frozen_flag"] = group_interp["pt903_frozen_flag"].fillna(0).astype(int)
    group_interp["bucket_id"] = bid

    all_buckets.append(group_interp)

print(f"Nr of buckets created: {len(all_buckets)}")

output_dir = "../data/processed/clean"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

count_exported = 0
for i, bucket in enumerate(all_buckets):
    if len(bucket) >= 1000:
        start_time = bucket.index.min().strftime('%Y-%m-%d_%H-%M')
        end_time = bucket.index.max().strftime('%Y-%m-%d_%H-%M')

        filename = f"data_session_{i}_{start_time}_to_{end_time}.csv"
        file_path = os.path.join(output_dir, filename)

        bucket.to_csv(file_path)
        print(f"Generated the file {filename} ({len(bucket)} observations)")
        count_exported += 1
    else:
        print(f"Ignored: Session {i} (only {len(bucket)} observations)")

print(f"{count_exported} files exported.")

Nr of buckets created: 18
Ignored: Session 0 (only 52 observations)
Generated the file data_session_1_2026-03-12_15-38_to_2026-03-14_15-57.csv (2900 observations)
Generated the file data_session_2_2026-03-14_22-50_to_2026-03-16_04-48.csv (1799 observations)
Generated the file data_session_3_2026-03-16_09-02_to_2026-03-19_16-24.csv (4763 observations)
Ignored: Session 4 (only 539 observations)
Generated the file data_session_5_2026-03-20_03-25_to_2026-03-21_06-10.csv (1606 observations)
Ignored: Session 6 (only 560 observations)
Ignored: Session 7 (only 6 observations)
Generated the file data_session_8_2026-03-21_18-10_to_2026-03-22_14-44.csv (1235 observations)
Ignored: Session 9 (only 102 observations)
Ignored: Session 10 (only 164 observations)
Ignored: Session 11 (only 4 observations)
Generated the file data_session_12_2026-03-23_14-22_to_2026-03-25_08-57.csv (2556 observations)
Generated the file data_session_13_2026-03-25_09-31_to_2026-03-31_14-28.csv (8938 observations)
Generated